# CV07 — Biến ảnh thành token rời rạc

Encoder giảm kích thước 4 lần mỗi chiều. Mỗi vector được thay bằng codebook entry gần nhất. Straight-through chuyển gradient reconstruction về encoder; hai VQ terms học codebook và commitment.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Encoder giảm kích thước 4 lần mỗi chiều. Mỗi vector được thay bằng codebook entry gần nhất. Straight-through chuyển gradient reconstruction về encoder; hai VQ terms học codebook và commitment. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv07_tokenization import VQVAE, loss_fn
images = torch.rand(2, 3, 32, 32, device=device)
model = VQVAE(d=32, k=64).to(device)
batch = (images,)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
with torch.no_grad():
    reconstruction, indices, vq_loss = model(images)
print('Reconstruction:', reconstruction.shape)
print('Raster token indices:', indices.shape)
print('VQ loss:', vq_loss.item())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Continuous AE → VQ-VAE → causal LM trên raster tokens → residual quantization/perceptual loss. Theo dõi dead codes, không chỉ PSNR.


In [ ]:
model.eval()
with torch.no_grad():
    reconstruction, indices, _ = model(images)
    counts = torch.bincount(indices.flatten(), minlength=model.code.num_embeddings).float()
    probabilities = counts / counts.sum()
    perplexity = (-(probabilities * probabilities.clamp_min(1e-9).log()).sum()).exp()
print('Used codebook entries:', (counts > 0).sum().item())
print('Fixture codebook perplexity:', perplexity.item())
print('Next: train a causal token LM on indices; optional transformers recipe in library_recipes.py.')


## Thử nghiệm tiếp theo

Continuous AE → VQ-VAE → causal LM trên raster tokens → residual quantization/perceptual loss. Theo dõi dead codes, không chỉ PSNR.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv07_tokenization.py) và [tài liệu gốc](../REFERENCE.md).
